In [1]:
import pandas as pd
import numpy as np

In [2]:
ratings_csv = r"C:\Users\fkamdar\Desktop\repos\pd_nonmotor\stimuli\all_NAPs_ratings.csv"
out_csv = r"C:\Users\fkamdar\Desktop\repos\pd_nonmotor\stimuli\session2_block_112.csv"
old_csv = r"C:\Users\fkamdar\Desktop\repos\pd_nonmotor\stimuli\blocks\pndm01_block_112.csv"

In [3]:
N_BINS = 8
TRIALS_PER_BIN = 14
COND_PER_BIN = 7
SEED = 42
np.random.seed(SEED)

In [4]:
# ===== LOAD DATA =====
df = pd.read_csv(ratings_csv)
old_df = pd.read_csv(old_csv)

# Updated df to exclude images in the old session1 csv block
df = df[~df['ID'].isin(old_df['ID'])]

# save the new updated df as csv


In [5]:
all_rating_postSession1_csv = ratings_csv.replace(".csv", "_postSession1.csv")
df.to_csv(all_rating_postSession1_csv, index=False)


In [ ]:
import pandas as pd

bins = [1, 2, 3, 4, 5, 6, 7, 8, 9]
bin_labels = [f"{bins[i]}-{bins[i+1]}" for i in range(len(bins) - 1)]

old_df["val_bin"] = pd.cut(
    old_df["Valence"],
    bins=bins,
    labels=bin_labels,
    include_lowest=True
)

counts = old_df["val_bin"].value_counts().reindex(bin_labels)
print(counts)

In [ ]:
# assign valence bins
bins = [1,2,3,4,5,6,7,8,9]
bin_labels = [f"{bins[i]}-{bins[i+1]}" for i in range(len(bins)-1)]
df["val_bin"] = pd.cut(df["Valence"], bins=bins, labels=bin_labels, include_lowest=True)
selected = []

In [ ]:
for b in bin_labels:

    pool = df[df["val_bin"] == b].copy()
    if pool["Category"].isna().any():
        n_missing = int(pool["Category"].isna().sum())
        raise ValueError(f"Bin {b} has {n_missing} missing Category values")

    if len(pool) < TRIALS_PER_BIN:
        raise ValueError(f"Bin {b} has only {len(pool)} images")

    cats = pool["Category"].unique().tolist()
    rng = np.random.default_rng(SEED + bin_labels.index(b))
    rng.shuffle(cats)

    available_counts = pool["Category"].value_counts().to_dict()
    target_counts = {cat: 0 for cat in cats}
    remaining = TRIALS_PER_BIN

    while remaining > 0:
        progressed = False
        for cat in cats:
            if target_counts[cat] < available_counts[cat]:
                target_counts[cat] += 1
                remaining -= 1
                progressed = True
                if remaining == 0:
                    break
        if not progressed:
            raise ValueError(f"Bin {b} cannot satisfy category-balance constraint")

    sampled_parts = []
    for cat in cats:
        n_take = target_counts[cat]
        if n_take > 0:
            cat_pool = pool[pool["Category"] == cat]
            sampled_parts.append(cat_pool.sample(n_take, random_state=SEED + bin_labels.index(b)))

    sample = pd.concat(sampled_parts).sample(frac=1, random_state=SEED + bin_labels.index(b)).reset_index(drop=True)

    sample = sample.copy()
    sample["condition"] = ["FEEL"] * COND_PER_BIN + ["TONE"] * COND_PER_BIN

    selected.append(sample)

In [ ]:
block = pd.concat(selected)
block = block.sample(frac=1).reset_index(drop=True)

In [ ]:
block["trial"] = np.arange(1, len(block)+1)

# filename column
block["filename"] = block["ID"] + ".jpg"

block[["trial","ID","filename","Category","Valence","val_bin","condition"]].to_csv(out_csv,index=False)

print("Block generated:", out_csv)
print(block.groupby(["val_bin","condition"]).size())